# Pneumonia Classification from Chest X-rays

**Author:** Blessing Osuagwu  
**Project type:** Educational deep-learning experiment

This project fine-tunes a MobileNetV2 image classifier to distinguish the two labels in a public chest X-ray dataset. The original project title mentioned Charité, but the cited dataset is the Kaggle chest X-ray pneumonia dataset; this repository does not establish a Charité affiliation or use Charité patient data, so this cleaned version removes that implication.

> **Important:** This is an educational experiment, not a medical device or diagnostic tool. It must not be used to make healthcare decisions.

## Dataset and use

The notebook downloads the dataset identified in the original project from Kaggle: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Review the source's license and terms. The images are not stored in this repository.

## 1. Setup and data

Install `requirements.txt` in a notebook runtime. The KaggleHub download may require Kaggle authentication or acceptance of dataset terms.

In [ ]:
from pathlib import Path
import random

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"

download_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
candidates = [download_dir / "chest_xray", download_dir]
dataset_dir = next((p for p in candidates if (p / "train" / "NORMAL").is_dir()), None)
if dataset_dir is None:
    raise FileNotFoundError(f"Could not locate train/NORMAL under {download_dir}")
print("Dataset:", dataset_dir)
print("Class order will be checked from the generators before evaluation.")

## 2. Training and validation data

The supplied validation directory contains only 16 images in this dataset version. This notebook instead reserves 20% of the supplied training directory for validation and keeps the supplied test directory untouched until final evaluation. The dataset does not provide documented patient IDs here, so patient-level independence cannot be verified.

In [ ]:
train_dir = dataset_dir / "train"
test_dir = dataset_dir / "test"
if not test_dir.is_dir():
    raise FileNotFoundError(f"Expected test directory: {test_dir}")

augment = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.20,
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True,
)
plain = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.20)

train_gen = augment.flow_from_directory(
    train_dir, target_size=IMG_SIZE, color_mode="rgb", class_mode="binary",
    batch_size=BATCH_SIZE, subset="training", seed=SEED, shuffle=True,
)
val_gen = plain.flow_from_directory(
    train_dir, target_size=IMG_SIZE, color_mode="rgb", class_mode="binary",
    batch_size=BATCH_SIZE, subset="validation", seed=SEED, shuffle=False,
)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
    test_dir, target_size=IMG_SIZE, color_mode="rgb", class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=False,
)

print("Class mapping:", train_gen.class_indices)
if train_gen.class_indices != {"NORMAL": 0, "PNEUMONIA": 1}:
    raise ValueError(f"Unexpected class mapping: {train_gen.class_indices}")

## 3. Model

MobileNetV2 is initialized with ImageNet weights. The Keras preprocessing function expected by MobileNetV2 is applied to RGB inputs. The head is intentionally small; this is a baseline transfer-learning model, not a clinically validated system.

In [ ]:
base = MobileNetV2(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)
model.summary()

## 4. Train

The validation subset is drawn only from the training directory. The held-out test directory is not used for model selection.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_pneumonia_model.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(
    train_gen, validation_data=val_gen, epochs=20, callbacks=callbacks,
    class_weight=dict(zip(
        np.unique(train_gen.classes),
        compute_class_weight(class_weight="balanced", classes=np.unique(train_gen.classes), y=train_gen.classes),
    )),
)

## 5. Final held-out evaluation

Run this section once after choosing the model. The positive class is `PNEUMONIA` (label 1). Accuracy alone is insufficient for this imbalanced dataset; inspect both class-level results and the confusion matrix.

In [ ]:
test_gen.reset()
probabilities = model.predict(test_gen, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype(int)
truth = test_gen.classes

print(classification_report(truth, predictions, target_names=["NORMAL", "PNEUMONIA"], digits=4))
cm = confusion_matrix(truth, predictions, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NORMAL", "PNEUMONIA"],
            yticklabels=["NORMAL", "PNEUMONIA"])
plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title("Held-out test confusion matrix")
plt.tight_layout(); plt.show()
print("ROC-AUC:", round(roc_auc_score(truth, probabilities), 4))

## 6. Save the model and limitations

Only save and publish weights after reviewing dataset license terms. Performance on this dataset does not establish generalization to other hospitals, ages, scanners, or clinical populations. No patient-level split is documented in the notebook, the dataset labels may contain noise, and this project has not undergone clinical validation or regulatory review.

In [ ]:
model.save("pneumonia_mobilenetv2.keras")
print("Saved pneumonia_mobilenetv2.keras")